In [2]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from torch.optim import AdamW

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from tqdm.auto import tqdm
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)
from google.colab import drive
drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/dataset_sir.zip"
import zipfile

extract_path = "/content/data"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Done!")


train_df=pd.read_csv(f"/content/data/telugu_train.csv")
val_df=pd.read_csv(f"/content/data/codemixed_validation.csv")
test_df=pd.read_csv(f"/content/data/codemixed_test.csv")

def clean_text(text):

    text=str(text)

    text=re.sub(r"http\S+"," ",text)

    text=re.sub(r"www\S+"," ",text)

    text=re.sub(r"@\w+"," ",text)

    text=re.sub(r"#"," ",text)

    text=re.sub(r"\s+"," ",text)

    return text.strip()
train_df["text"]=train_df["text"].apply(clean_text)

val_df["text"]=val_df["text"].apply(clean_text)

test_df["text"]=test_df["text"].apply(clean_text)

train_df=train_df.drop_duplicates(subset=["text"])

val_df=val_df.drop_duplicates(subset=["text"])

test_df=test_df.drop_duplicates(subset=["text"])

labels=sorted(train_df.label.unique())
MODEL_NAME="ai4bharat/IndicBERTv2-MLM-only"

tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)

class TeluguDataset(Dataset):

    def __init__(self,df,tokenizer,max_len=128):

        self.texts=df.text.tolist()

        self.labels=df.label.tolist()

        self.tokenizer=tokenizer

        self.max_len=max_len

    def __len__(self):

        return len(self.texts)

    def __getitem__(self,idx):

        encoding=self.tokenizer(

            self.texts[idx],

            truncation=True,

            padding="max_length",

            max_length=self.max_len,

            return_tensors="pt"

        )

        return {

            "input_ids":encoding["input_ids"].squeeze(),

            "attention_mask":encoding["attention_mask"].squeeze(),

            "labels":torch.tensor(self.labels[idx],dtype=torch.long)

        }


train_dataset=TeluguDataset(train_df,tokenizer)

val_dataset=TeluguDataset(val_df,tokenizer)

test_dataset=TeluguDataset(test_df,tokenizer)


train_loader=DataLoader(train_dataset,batch_size=16,shuffle=True)

val_loader=DataLoader(val_dataset,batch_size=16)

test_loader=DataLoader(test_dataset,batch_size=16)


model=AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=2
)

model.to(device)
optimizer=AdamW(

    model.parameters(),

    lr=2e-5,

    weight_decay=0.01
)
epochs=5

total_steps=len(train_loader)*epochs

scheduler=get_linear_schedule_with_warmup(

    optimizer,

    num_warmup_steps=0,

    num_training_steps=total_steps
)

best_f1=0

for epoch in range(epochs):

    model.train()

    train_loss=0

    loop=tqdm(train_loader)

    for batch in loop:

        optimizer.zero_grad()

        input_ids=batch["input_ids"].to(device)

        attention=batch["attention_mask"].to(device)

        labels=batch["labels"].to(device)

        outputs=model(

            input_ids=input_ids,

            attention_mask=attention,

            labels=labels

        )

        loss=outputs.loss

        loss.backward()

        optimizer.step()

        scheduler.step()

        train_loss+=loss.item()

        loop.set_description(f"Epoch {epoch+1}")

        loop.set_postfix(loss=loss.item())

    #########################

    model.eval()

    predictions=[]

    true_labels=[]

    val_loss=0

    with torch.no_grad():

        for batch in val_loader:

            input_ids=batch["input_ids"].to(device)

            attention=batch["attention_mask"].to(device)

            labels=batch["labels"].to(device)

            outputs=model(

                input_ids=input_ids,

                attention_mask=attention,

                labels=labels

            )

            val_loss+=outputs.loss.item()

            preds=torch.argmax(outputs.logits,dim=1)

            predictions.extend(preds.cpu().numpy())

            true_labels.extend(labels.cpu().numpy())

    acc=accuracy_score(true_labels,predictions)

    p,r,f1,_=precision_recall_fscore_support(

        true_labels,

        predictions,

        average="macro"

    )

    print()

    print("Epoch:",epoch+1)

    print("Train Loss:",train_loss/len(train_loader))

    print("Val Loss:",val_loss/len(val_loader))

    print("Accuracy:",acc)

    print("Macro F1:",f1)

    if f1>best_f1:

        best_f1=f1

        torch.save(model.state_dict(),"best_model.pt")

        print("Best model saved")


model.load_state_dict(torch.load("best_model.pt"))

model.eval()

predictions=[]

true_labels=[]

with torch.no_grad():

    for batch in test_loader:

        input_ids=batch["input_ids"].to(device)

        attention=batch["attention_mask"].to(device)

        labels=batch["labels"].to(device)

        outputs=model(

            input_ids=input_ids,

            attention_mask=attention

        )

        preds=torch.argmax(outputs.logits,dim=1)

        predictions.extend(preds.cpu().numpy())

        true_labels.extend(labels.cpu().numpy())


acc=accuracy_score(true_labels,predictions)

precision,recall,f1,_=precision_recall_fscore_support(

    true_labels,

    predictions,

    average="macro"

)

print(f"Accuracy : {acc:.4f}")

print(f"Precision: {precision:.4f}")

print(f"Recall   : {recall:.4f}")

print(f"Macro F1 : {f1:.4f}")



print(classification_report(
    true_labels,
    predictions,
    target_names=["Class 0", "Class 1"]
))
cm=confusion_matrix(true_labels,predictions)

print(cm)

cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Done!


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-only
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those p

  0%|          | 0/1498 [00:00<?, ?it/s]


Epoch: 1
Train Loss: 0.28800550704506117
Val Loss: 0.6300918889045716
Accuracy: 0.6196473551637279
Macro F1: 0.5990408604164298
Best model saved


...


Epoch: 3
Train Loss: 0.13892018133202427
Val Loss: 0.8030926275253296
Accuracy: 0.6700251889168766
Macro F1: 0.6647067546048263


  0%|          | 0/1498 [00:00<?, ?it/s]


Epoch: 4
Train Loss: 0.08905389507893176
Val Loss: 0.9793236696720123
Accuracy: 0.6801007556675063
Macro F1: 0.6774484527838376


  0%|          | 0/1498 [00:00<?, ?it/s]


Epoch: 5
Train Loss: 0.05595619731069168
Val Loss: 1.1183867609500886
Accuracy: 0.6801007556675063
Macro F1: 0.6764798357342231
Accuracy : 0.7200
Precision: 0.7202
Recall   : 0.7190
Macro F1 : 0.7192
              precision    recall  f1-score   support

     Class 0       0.72      0.69      0.70       194
     Class 1       0.72      0.75      0.73       206

    accuracy                           0.72       400
   macro avg       0.72      0.72      0.72       400
weighted avg       0.72      0.72      0.72       400

[[133  61]
 [ 51 155]]
